In [1]:
print("="*70)
print("MDSS v2.0 - Colab-Compatible Version")
print("="*70)
print("\n📦 Installing dependencies (Colab-optimized)...\n")

# Strategy: Work WITH Colab's pre-installed versions, not against them
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Use Colab's existing numpy/pandas/sklearn (don't reinstall!)
# Only install what's missing
try:
    install("openai-whisper")
    install("sentence-transformers")
    install("sumy")
    install("python-louvain")
    install("google-generativeai")
    install("gradio")

    # Download spaCy model
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "--quiet"])

    print("✅ Dependencies installed!\n")
except Exception as e:
    print(f"⚠️ Installation warning: {e}")
    print("Continuing with available packages...\n")



MDSS v2.0 - Colab-Compatible Version

📦 Installing dependencies (Colab-optimized)...

✅ Dependencies installed!



In [2]:
# ============================================================================
# IMPORTS
# ============================================================================
import warnings
warnings.filterwarnings('ignore')

import os, re, nltk, torch, traceback
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
from datetime import datetime
from collections import defaultdict, Counter

# Try importing sklearn - if it fails, we'll use alternatives
try:
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.cluster import AgglomerativeClustering
    SKLEARN_AVAILABLE = True
except ImportError:
    print("⚠️ sklearn unavailable, using numpy alternatives")
    SKLEARN_AVAILABLE = False

    # Numpy-based cosine similarity
    def cosine_similarity(X, Y=None):
        if Y is None:
            Y = X
        X = np.array(X)
        Y = np.array(Y)
        X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
        Y_norm = Y / np.linalg.norm(Y, axis=1, keepdims=True)
        return np.dot(X_norm, Y_norm.T)

    # Simple clustering alternative
    class AgglomerativeClustering:
        def __init__(self, n_clusters=2, **kwargs):
            self.n_clusters = n_clusters

        def fit_predict(self, X):
            # Simple k-means-like clustering
            n_samples = len(X)
            if n_samples <= self.n_clusters:
                return np.arange(n_samples)

            # Random initialization
            np.random.seed(42)
            centers_idx = np.random.choice(n_samples, self.n_clusters, replace=False)

            for _ in range(10):  # 10 iterations
                # Assign to nearest center
                similarities = cosine_similarity(X, X[centers_idx])
                labels = np.argmax(similarities, axis=1)

                # Update centers
                for i in range(self.n_clusters):
                    mask = labels == i
                    if mask.any():
                        centers_idx[i] = np.where(mask)[0][0]

            return labels

from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import community.community_louvain as community_louvain

import whisper
import google.generativeai as genai
import gradio as gr
import spacy

print("✅ All imports completed\n")


✅ All imports completed



In [3]:
# ============================================================================
# NLTK SETUP
# ============================================================================
print("📚 Setting up NLTK...")
for resource in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource, quiet=True)
print("✅ NLTK ready\n")

# Load spaCy
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy loaded\n")
except:
    print("⚠️ spaCy model not loaded (optional)\n")
    nlp = None


📚 Setting up NLTK...
✅ NLTK ready

✅ spaCy loaded



In [4]:
# ============================================================================
# CONFIGURATION
# ============================================================================
class MDSSConfig:
    WHISPER_MODEL = "tiny"  # Use tiny for faster loading in Colab
    EMBEDDING_MODEL = "all-MiniLM-L6-v2"
    SARCASM_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    MIN_MODULE_SIZE = 2
    EXTRACTIVE_SENTENCES = 3
    REDUNDANCY_THRESHOLD = 0.85
    NUM_SPEAKERS = 2

config = MDSSConfig()


In [5]:
# ============================================================================
# GEMINI SETUP
# ============================================================================
def setup_gemini_api():
    print("="*70)
    print("🔑 GEMINI API SETUP")
    print("="*70)
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY not found")

        genai.configure(api_key=api_key)

        # Try to find any available model
        try:
            available_models = [m.name for m in genai.list_models()
                              if 'generateContent' in m.supported_generation_methods]
            if not available_models:
                raise RuntimeError("No Gemini models available")

            # Prefer flash models
            model_name = None
            for prefix in ['models/gemini-2.0-flash', 'models/gemini-1.5-flash', 'models/gemini']:
                for m in available_models:
                    if m.startswith(prefix):
                        model_name = m
                        break
                if model_name:
                    break

            if not model_name:
                model_name = available_models[0]

        except Exception as e:
            print(f"⚠️ Model listing failed: {e}")
            model_name = "models/gemini-1.5-flash"  # Fallback

        # Test connection
        model = genai.GenerativeModel(model_name)
        response = model.generate_content("Say 'ready'")
        print(f"✅ Gemini connected: {model_name}")
        print(f"   Test: {response.text.strip()}\n")
        return True, model_name

    except Exception as e:
        print(f"\n❌ GEMINI SETUP FAILED: {e}")
        print("\n📋 To fix:")
        print("   1. Click 🔑 in left sidebar")
        print("   2. Add: GEMINI_API_KEY")
        print("   3. Get key: https://aistudio.google.com/apikey\n")
        return False, None

GEMINI_READY, GEMINI_MODEL = setup_gemini_api()
if not GEMINI_READY:
    raise RuntimeError("Set up Gemini API key first!")

print("="*70 + "\n")



🔑 GEMINI API SETUP
✅ Gemini connected: models/gemini-2.0-flash-exp
   Test: Ready.




In [6]:
# ============================================================================
# SPEAKER DIARIZATION
# ============================================================================
class SpeakerDiarization:
    def __init__(self):
        print("🔧 Initializing Speaker Diarization...")
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)
        print("✅ Ready\n")

    def diarize(self, utterances, num_speakers=None):
        if not utterances or len(utterances) < 2:
            return [{"speaker": "SPEAKER_0", "text": u} for u in utterances]

        print(f"🎭 Diarizing {len(utterances)} utterances...")

        embeddings = self.embedder.encode(utterances, show_progress_bar=False)

        if num_speakers is None:
            num_speakers = min(config.NUM_SPEAKERS, len(utterances))

        clustering = AgglomerativeClustering(n_clusters=num_speakers)
        labels = clustering.fit_predict(embeddings)

        # Merge consecutive same-speaker utterances
        diarized = []
        current_speaker = None
        current_text = []

        for utt, label in zip(utterances, labels):
            speaker = f"SPEAKER_{label}"
            if speaker == current_speaker:
                current_text.append(utt)
            else:
                if current_text:
                    diarized.append({
                        "speaker": current_speaker,
                        "text": " ".join(current_text)
                    })
                current_speaker = speaker
                current_text = [utt]

        if current_text:
            diarized.append({
                "speaker": current_speaker,
                "text": " ".join(current_text)
            })

        print(f"✓ Found {len(set(labels))} speakers\n")
        return diarized


In [7]:
# ============================================================================
# SARCASM DETECTOR
# ============================================================================
class SarcasmDetector:
    def __init__(self):
        print("🔧 Initializing Sarcasm Detector...")
        try:
            self.detector = pipeline(
                "sentiment-analysis",
                model=config.SARCASM_MODEL,
                device=0 if config.DEVICE == 'cuda' else -1
            )
            print("✅ Ready\n")
        except Exception as e:
            print(f"⚠️ Unavailable: {e}\n")
            self.detector = None

    def detect(self, text):
        if not self.detector or len(text.strip()) < 5:
            return {'is_sarcastic': False, 'score': 0.0}

        try:
            result = self.detector(text[:512])[0]
            label = result['label'].lower()
            score = result['score']
            is_sarcastic = 'negative' in label and score > 0.8
            return {'is_sarcastic': is_sarcastic, 'score': score if is_sarcastic else 0.0}
        except:
            return {'is_sarcastic': False, 'score': 0.0}


In [8]:
# ============================================================================
# GEMINI SUMMARIZER
# ============================================================================
class GeminiSummarizer:
    def __init__(self):
        print("🔧 Initializing Gemini...")
        self.model = genai.GenerativeModel(GEMINI_MODEL)
        print("✅ Ready\n")

    def summarize(self, text, context="dialogue", max_length=200):
        if not text or len(text.strip()) < 10:
            return "Text too short."

        prompt = f"Summarize concisely (max {max_length} words):\n\n{text}\n\nSummary:"
        try:
            response = self.model.generate_content(prompt)
            return response.text.strip() or "Failed."
        except Exception as e:
            print(f"⚠️ Error: {e}")
            return "Summary failed."

    def classify_intent_hierarchical(self, text):
        prompt = f"""Classify intent:
PRIMARY: INQUIRY|COMPLAINT|REQUEST|FEEDBACK|STATEMENT|OTHER
SECONDARY: Information/Service/Action

Text: {text[:300]}

Format: PRIMARY|SECONDARY
Intent:"""
        try:
            response = self.model.generate_content(prompt).text.strip().upper()
            if '|' in response:
                primary, secondary = response.split('|', 1)
                return {'primary': primary.strip(), 'secondary': secondary.strip()}
            return {'primary': response.strip(), 'secondary': 'GENERAL'}
        except:
            return {'primary': 'STATEMENT', 'secondary': 'GENERAL'}

    def check_consistency(self, summary):
        prompt = f"Check for contradictions. Reply: CONSISTENT or INCONSISTENT|reason\n\n{summary}\n\nResult:"
        try:
            response = self.model.generate_content(prompt).text.strip()
            if 'CONSISTENT' in response and 'INCONSISTENT' not in response:
                return {'consistent': True, 'issues': None}
            parts = response.split('|', 1)
            return {'consistent': False, 'issues': parts[1] if len(parts) > 1 else "Unknown"}
        except:
            return {'consistent': True, 'issues': None}

    def correct_grammar(self, text):
        prompt = f"Correct grammar (return corrected text only):\n\n{text}\n\nCorrected:"
        try:
            return self.model.generate_content(prompt).text.strip()
        except:
            return text

    def assemble_final_summary(self, module_summaries):
        if len(module_summaries) == 1:
            return list(module_summaries.values())[0]['abstractive']

        modules_text = "\n".join([
            f"{i+1}. {s['primary_intent']}: {s['abstractive']}"
            for i, s in enumerate(module_summaries.values())
        ])

        prompt = f"Combine into coherent summary:\n\n{modules_text}\n\nFinal:"
        try:
            return self.model.generate_content(prompt).text.strip()
        except:
            return "Assembly failed."


In [9]:
# ============================================================================
# INTENT GRAPH
# ============================================================================
class IntentGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.node_id = 0

    def add_utterance(self, text, intent, speaker, sarcasm_info, embedding):
        self.graph.add_node(
            self.node_id,
            text=text,
            intent=intent['primary'],
            speaker=speaker,
            is_sarcastic=sarcasm_info['is_sarcastic'],
            embedding=embedding
        )

        if self.node_id > 0:
            prev_emb = self.graph.nodes[self.node_id - 1]['embedding']
            sim = cosine_similarity([prev_emb], [embedding])[0][0]
            self.graph.add_edge(self.node_id - 1, self.node_id, weight=float(sim))

        self.node_id += 1

    def visualize(self, save_path="intent_graph.png"):
        if len(self.graph.nodes()) == 0:
            return

        try:
            plt.figure(figsize=(12, 8))
            pos = nx.spring_layout(self.graph, k=1.5, iterations=30)

            colors = {'INQUIRY': '#FF6B6B', 'COMPLAINT': '#FF8B94',
                     'REQUEST': '#FFE66D', 'FEEDBACK': '#FCBAD3',
                     'STATEMENT': '#4ECDC4', 'OTHER': '#CCC'}

            node_colors = [colors.get(self.graph.nodes[n]['intent'], '#CCC')
                          for n in self.graph.nodes()]
            node_sizes = [700 if self.graph.nodes[n]['is_sarcastic'] else 400
                         for n in self.graph.nodes()]

            nx.draw_networkx_nodes(self.graph, pos, node_color=node_colors,
                                 node_size=node_sizes, alpha=0.7)
            nx.draw_networkx_edges(self.graph, pos, alpha=0.2, arrows=True)

            labels = {n: f"{n}" for n in self.graph.nodes()}
            nx.draw_networkx_labels(self.graph, pos, labels, font_size=7)

            plt.title("Intent Graph", fontsize=12)
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            print(f"✅ Graph saved: {save_path}")
            plt.close()
        except Exception as e:
            print(f"⚠️ Visualization failed: {e}")

    def cluster_graph(self):
        print("🔬 Clustering...")

        if len(self.graph.nodes()) < config.MIN_MODULE_SIZE:
            return {0: list(self.graph.nodes())}

        try:
            G_und = self.graph.to_undirected()
            partition = community_louvain.best_partition(G_und, resolution=1.0)

            modules = defaultdict(list)
            for node, cid in partition.items():
                modules[cid].append(node)

            modules = {k: v for k, v in modules.items()
                      if len(v) >= config.MIN_MODULE_SIZE}

            if not modules:
                modules = {0: list(self.graph.nodes())}

            print(f"✓ {len(modules)} modules\n")
            return modules
        except Exception as e:
            print(f"⚠️ Clustering failed: {e}, using single module")
            return {0: list(self.graph.nodes())}

In [10]:
# ============================================================================
# HIERARCHICAL SUMMARIZER
# ============================================================================
class HierarchicalSummarizer:
    def __init__(self, gemini):
        self.gemini = gemini
        self.textrank = TextRankSummarizer()
        self.tokenizer = Tokenizer("english")
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)

    def _extractive_summarize(self, text):
        try:
            parser = PlaintextParser.from_string(text, self.tokenizer)
            summary = self.textrank(parser.document, sentences_count=config.EXTRACTIVE_SENTENCES)
            return " ".join(str(s) for s in summary)
        except:
            sentences = nltk.sent_tokenize(text)
            return " ".join(sentences[:config.EXTRACTIVE_SENTENCES])

    def summarize_module(self, graph, nodes):
        texts = []
        for node in nodes:
            data = graph.nodes[node]
            text = data['text']
            if data['is_sarcastic']:
                text = f"[SARCASTIC] {text}"
            texts.append(text)

        combined = " ".join(texts)
        extractive = self._extractive_summarize(combined)
        abstractive = self.gemini.summarize(extractive, "module")

        intents = [graph.nodes[n]['intent'] for n in nodes]
        primary = Counter(intents).most_common(1)[0][0]

        return {
            'extractive': extractive,
            'abstractive': abstractive,
            'primary_intent': primary
        }

In [11]:
# ============================================================================
# REDUNDANCY REMOVER
# ============================================================================
class RedundancyRemover:
    def __init__(self):
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)

    def remove_redundancy(self, text):
        sentences = nltk.sent_tokenize(text)
        if len(sentences) <= 1:
            return text

        print(f"🧹 Removing redundancy...")
        embeddings = self.embedder.encode(sentences, show_progress_bar=False)

        unique_sent = [sentences[0]]
        unique_emb = [embeddings[0]]

        for i in range(1, len(sentences)):
            sims = cosine_similarity([embeddings[i]], unique_emb)[0]
            if max(sims) < config.REDUNDANCY_THRESHOLD:
                unique_sent.append(sentences[i])
                unique_emb.append(embeddings[i])

        print(f"✓ {len(unique_sent)} unique sentences\n")
        return " ".join(unique_sent)


In [12]:
# ============================================================================
# MDSS SYSTEM
# ============================================================================
class MDSS:
    def __init__(self):
        print("\n" + "="*70)
        print("🚀 Initializing MDSS v2.0")
        print("="*70 + "\n")

        self.diarizer = SpeakerDiarization()
        self.sarcasm_detector = SarcasmDetector()
        self.gemini = GeminiSummarizer()
        self.redundancy_remover = RedundancyRemover()
        self.embedder = SentenceTransformer(config.EMBEDDING_MODEL, device=config.DEVICE)

        print(f"Loading Whisper ({config.WHISPER_MODEL})...")
        self.whisper = whisper.load_model(config.WHISPER_MODEL)
        print("✅ Whisper loaded\n")

        print("="*70)
        print("✅ MDSS Ready")
        print("="*70 + "\n")

    def transcribe_audio(self, audio_path):
        print(f"🎧 Transcribing...")
        result = self.whisper.transcribe(audio_path)
        print(f"✅ Done\n")

        if 'segments' in result:
            return [seg['text'].strip() for seg in result['segments']]
        return nltk.sent_tokenize(result['text'])

    def analyze_dialogue(self, dialogue_input, is_audio=False):
        print("\n" + "="*70)
        print("📊 MDSS ANALYSIS")
        print("="*70 + "\n")

        # Get utterances
        if is_audio:
            utterances = self.transcribe_audio(dialogue_input)
        else:
            utterances = nltk.sent_tokenize(dialogue_input)

        # Diarization
        print("Step 1: Speaker Diarization")
        diarized = self.diarizer.diarize(utterances)

        # Build graph
        print("Step 2: Intent Graph")
        intent_graph = IntentGraph()

        for entry in diarized:
            text = entry['text']
            speaker = entry['speaker']
            sarcasm = self.sarcasm_detector.detect(text)
            intent = self.gemini.classify_intent_hierarchical(text)
            embedding = self.embedder.encode(text, show_progress_bar=False)
            intent_graph.add_utterance(text, intent, speaker, sarcasm, embedding)

        print(f"✓ {len(intent_graph.graph.nodes())} nodes\n")

        # Cluster
        print("Step 3: Clustering")
        modules = intent_graph.cluster_graph()

        # Summarize
        print("Step 4: Summarization")
        summarizer = HierarchicalSummarizer(self.gemini)
        module_summaries = {}

        for mid, nodes in modules.items():
            print(f"  Module {mid}...")
            module_summaries[mid] = summarizer.summarize_module(intent_graph.graph, nodes)

        # Assemble
        print("\nStep 5: Assembly")
        final = self.gemini.assemble_final_summary(module_summaries)

        # Polish
        print("Step 6: Polishing")
        final = self.redundancy_remover.remove_redundancy(final)
        consistency = self.gemini.check_consistency(final)
        final = self.gemini.correct_grammar(final)

        # Visualize
        intent_graph.visualize()

        print("\n" + "="*70)
        print("✅ COMPLETE")
        print("="*70 + "\n")

        return {
            'final_summary': final,
            'module_summaries': module_summaries,
            'diarized': diarized,
            'consistency': consistency,
            'num_modules': len(modules)
        }



In [13]:
# ============================================================================
# GRADIO INTERFACE
# ============================================================================
def create_interface():
    mdss = MDSS()

    def process(audio, text):
        try:
            if audio:
                results = mdss.analyze_dialogue(audio, is_audio=True)
                inp = "🎤 Audio"
            elif text and text.strip():
                results = mdss.analyze_dialogue(text, is_audio=False)
                inp = "📝 Text"
            else:
                return "❌ Provide input!"

            out = f"**{inp}**\n\n**Summary:**\n{results['final_summary']}\n\n"
            out += f"**Stats:**\n- Speakers: {len(set([d['speaker'] for d in results['diarized']]))}\n"
            out += f"- Modules: {results['num_modules']}\n"
            out += f"- Consistent: {'✅' if results['consistency']['consistent'] else '⚠️'}\n\n"

            out += "**Modules:**\n"
            for i, (_, s) in enumerate(results['module_summaries'].items(), 1):
                out += f"{i}. {s['primary_intent']}: {s['abstractive']}\n"

            return out
        except Exception as e:
            return f"❌ Error: {str(e)}\n{traceback.format_exc()}"

    return gr.Interface(
        fn=process,
        inputs=[
            gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎤 Audio"),
            gr.Textbox(lines=8, placeholder="Or paste text...", label="📝 Text")
        ],
        outputs=gr.Textbox(label="📊 Results", lines=20),
        title="🎯 MDSS v2.0 - Dialogue Summarization",
        description="Speaker Diarization • Sarcasm Detection • Intent Classification • Graph Clustering",
        theme="soft",
        examples=[[None, "A: How was it?\nB: Oh, wonderful. Three hours of nothing.\nA: That bad?\nB: Yes."]]
    )



In [14]:
# ============================================================================
# TEST
# ============================================================================
def test_mdss():
    print("🧪 Testing...\n")
    sample = """Hi, I'm calling about my order. It hasn't arrived.
I'm sorry. Let me check.
I ordered two weeks ago.
Can you provide your order number?
ORDER-12345.
There was a delay. I'll expedite a replacement.
Any charges?
No, with 20% discount.
Thank you."""

    try:
        mdss = MDSS()
        results = mdss.analyze_dialogue(sample)
        print("\n" + "="*70)
        print("TEST RESULTS")
        print("="*70)
        print(f"\n{results['final_summary']}\n")
        print("✅ SUCCESS!")
        return results
    except Exception as e:
        print(f"❌ Failed: {e}")
        traceback.print_exc()


In [15]:
# ============================================================================
# LAUNCH
# ============================================================================
print("\n" + "="*70)
print("✨ MDSS v2.0 READY")
print("="*70)
print("\nOptions:")
print("  • interface.launch() - Start UI")
print("  • test_mdss() - Run test")
print("\n🚀 LAUNCHING...\n")

interface = create_interface()
interface.launch(share=True, debug=True)

print("\n✅ RUNNING")
print("="*70)


✨ MDSS v2.0 READY

Options:
  • interface.launch() - Start UI
  • test_mdss() - Run test

🚀 LAUNCHING...


🚀 Initializing MDSS v2.0

🔧 Initializing Speaker Diarization...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

100%|██████████████████████████████████████| 72.1M/72.1M [00:00<00:00, 119MiB/s]


✅ Whisper loaded

✅ MDSS Ready

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a1cae4897d90c2d6b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



📊 MDSS ANALYSIS

Step 1: Speaker Diarization
🎭 Diarizing 5 utterances...
✓ Found 2 speakers

Step 2: Intent Graph
✓ 3 nodes

Step 3: Clustering
🔬 Clustering...
✓ 1 modules

Step 4: Summarization
  Module 0...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash-exp:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2923.00ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash-exp:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2661.42ms



Step 5: Assembly
Step 6: Polishing
🧹 Removing redundancy...
✓ 2 unique sentences

✅ Graph saved: intent_graph.png

✅ COMPLETE


📊 MDSS ANALYSIS

Step 1: Speaker Diarization
🎭 Diarizing 4 utterances...
✓ Found 2 speakers

Step 2: Intent Graph
✓ 3 nodes

Step 3: Clustering
🔬 Clustering...
✓ 1 modules

Step 4: Summarization
  Module 0...

Step 5: Assembly
Step 6: Polishing
🧹 Removing redundancy...
✓ 4 unique sentences

✅ Graph saved: intent_graph.png

✅ COMPLETE


📊 MDSS ANALYSIS

🎧 Transcribing...
✅ Done

Step 1: Speaker Diarization
🎭 Diarizing 6 utterances...
✓ Found 2 speakers

Step 2: Intent Graph
✓ 2 nodes

Step 3: Clustering
🔬 Clustering...
✓ 1 modules

Step 4: Summarization
  Module 0...

Step 5: Assembly
Step 6: Polishing
🧹 Removing redundancy...
✓ 4 unique sentences

✅ Graph saved: intent_graph.png

✅ COMPLETE

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a1cae4897d90c2d6b5.gradio.live

✅ RUNNING
